# 03 — Knapsack Engine v3: Meal Planner Dynamic Budget (min Rp10K)

## Perubahan dari v1/v2
- Budget **ditentukan user** (min Rp 10.000/hari)
- **Budget Tiering**: LOW (<20K), MID (20-50K), HIGH (>50K)
- **Multi-pass knapsack**: greedy → gap fill → fractional serving → doubling
- Calorie deviation avg ≤ 8.7% (vs 48.7% sebelumnya)

**Output:**
- `output/preprocessed/food_master_v3.parquet`
- `output/validation/validation_report_v3.json`
- `output/models/knapsack_config_v3.json`

In [21]:
import pandas as pd
import numpy as np
import json
import random
import os
from pathlib import Path

random.seed(42)
np.random.seed(42)

os.makedirs("output/preprocessed", exist_ok=True)
os.makedirs("output/validation", exist_ok=True)
os.makedirs("output/models", exist_ok=True)

df_food = pd.read_parquet("output/preprocessed/food_master.parquet")
print(f"Food master: {df_food.shape}")
df_food.head(3)

Food master: (1346, 19)


,id,slug,name,category,cuisine,calories_per_portion,protein_g,fat_g,carbs_g,estimated_price_idr,is_halal,is_vegetarian,is_vegan,is_gluten_free,image_url,base_portion,base_portion_gram,fiber_g,is_active
0,1,abon,Abon,STAPLE,INDONESIAN_GENERAL,280.0,9.2,28.4,0.0,12000,True,True,True,True,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...,1 porsi,100,0.0,True
1,2,abon-haruwan,Abon haruwan,STAPLE,INDONESIAN_GENERAL,513.0,23.7,37.0,21.3,18500,True,True,True,True,https://img-global.cpcdn.com/recipes/cbf330fbd...,1 porsi,100,0.0,True
2,3,agar-agar,Agar-agar,STAPLE,INDONESIAN_GENERAL,1.0,0.0,0.2,0.0,8500,True,True,True,True,https://res.cloudinary.com/dk0z4ums3/image/upl...,1 porsi,100,0.0,True


In [22]:
# ── Configuration ───────────────────────────────────────────────
GOAL_WEIGHTS = {
    "WEIGHT_LOSS":  {"protein": 0.40, "fiber": 0.30, "fat_penalty": 0.20, "cal_bonus": 0.10},
    "MUSCLE_GAIN":  {"protein": 0.50, "fiber": 0.10, "fat_penalty": 0.10, "cal_bonus": 0.30},
    "MAINTENANCE":  {"protein": 0.35, "fiber": 0.20, "fat_penalty": 0.20, "cal_bonus": 0.25},
    "PERFORMANCE":  {"protein": 0.40, "fiber": 0.10, "fat_penalty": 0.15, "cal_bonus": 0.35},
}

MEAL_SPLITS = {
    2: {"BREAKFAST": 0.40, "DINNER": 0.60},
    3: {"BREAKFAST": 0.28, "LUNCH": 0.40, "DINNER": 0.32},
    4: {"BREAKFAST": 0.22, "LUNCH": 0.35, "SNACK": 0.10, "DINNER": 0.33},
}

MEAL_CATEGORY_PREF = {
    "BREAKFAST": {"STAPLE": 1.5, "BEVERAGE": 1.3, "PROTEIN": 1.2, "FRUIT": 1.2},
    "LUNCH":     {"STAPLE": 1.0, "PROTEIN": 1.5, "VEGETABLE": 1.3},
    "DINNER":    {"PROTEIN": 1.5, "VEGETABLE": 1.3, "STAPLE": 1.0},
    "SNACK":     {"FRUIT": 1.5, "SNACK": 1.3, "BEVERAGE": 1.0},
}

PORTION_CAL_RANGE = {
    "STAPLE": (80, 600), "PROTEIN": (50, 500), "VEGETABLE": (20, 200),
    "FRUIT": (30, 250), "BEVERAGE": (10, 300), "SNACK": (50, 400), "DESSERT": (80, 500),
}

print("Config loaded")

Config loaded


In [23]:
# ── Preprocessing ────────────────────────────────────────────────
def preprocess_food_master(df):
    df = df.copy()
    if "is_active" not in df.columns:
        df["is_active"] = True
    for cat, (lo, hi) in PORTION_CAL_RANGE.items():
        mask = df["category"] == cat
        too_high = mask & (df["calories_per_portion"] > hi)
        if too_high.any():
            scale = hi / df.loc[too_high, "calories_per_portion"]
            for col in ["calories_per_portion", "protein_g", "fat_g", "carbs_g"]:
                if col in df.columns:
                    df.loc[too_high, col] = (df.loc[too_high, col] * scale).round(1)
            df.loc[too_high, "estimated_price_idr"] = (
                df.loc[too_high, "estimated_price_idr"] * scale * 0.8).astype(int)
    df.loc[df["calories_per_portion"] < 15, "is_active"] = False
    df["estimated_price_idr"] = df["estimated_price_idr"].clip(lower=500)
    df["cal_per_1000_idr"] = (df["calories_per_portion"] / df["estimated_price_idr"] * 1000).round(1)
    return df

df_food = preprocess_food_master(df_food)
active = df_food[df_food["is_active"] == True]
print(f"Active items   : {len(active)}")
print(f"Price range    : Rp{int(active['estimated_price_idr'].min()):,} — Rp{int(active['estimated_price_idr'].max()):,}")
print(f"Calorie range  : {active['calories_per_portion'].min():.0f} — {active['calories_per_portion'].max():.0f} kkal")

df_food.to_parquet("output/preprocessed/food_master_v3.parquet", index=False)
print("Saved: food_master_v3.parquet")

Active items   : 1334
Price range    : Rp3,281 — Rp40,000
Calorie range  : 15 — 600 kkal
Saved: food_master_v3.parquet


In [24]:
# ── Budget Tiering & Scoring ─────────────────────────────────────
def _get_budget_tier(budget_per_day):
    if budget_per_day < 20000: return "LOW"
    if budget_per_day < 50000: return "MID"
    return "HIGH"

def _score_food(food, goal, meal_type, budget_tier):
    w = GOAL_WEIGHTS.get(goal, GOAL_WEIGHTS["MAINTENANCE"])
    price = max(food.get("estimated_price_idr", 500), 500)
    cal   = max(food.get("calories_per_portion", 1), 1)
    protein_score = w["protein"]     * food.get("protein_g", 0) / price * 1000
    fiber_score   = w.get("fiber", 0.2) * food.get("fiber_g", 0) / price * 1000
    fat_penalty   = w["fat_penalty"] * food.get("fat_g", 0) / price * 500
    cal_eff       = w["cal_bonus"]   * cal / price * 100
    if budget_tier == "LOW":
        score = cal_eff * 2.0 + protein_score * 0.5
    elif budget_tier == "MID":
        score = cal_eff * 1.0 + protein_score * 1.0 + fiber_score * 0.5
    else:
        score = protein_score + fiber_score - fat_penalty + cal_eff
    cat  = food.get("category", "STAPLE")
    pref = MEAL_CATEGORY_PREF.get(meal_type, {})
    return max(score * pref.get(cat, 0.7), 0.001)

print("Budget tiering & scoring ready")

Budget tiering & scoring ready


In [25]:
# ── Knapsack v3 — Multi-pass ──────────────────────────────────────
def knapsack_v3(pool, budget, calorie_target, goal="MAINTENANCE",
                meal_type="LUNCH", budget_tier="MID", max_items=5):
    if pool.empty or budget <= 0 or calorie_target <= 0:
        return []
    budget_hard = int(budget * 1.20) if budget_tier == "LOW" else budget
    affordable  = pool[pool["estimated_price_idr"] <= budget_hard].copy()
    if affordable.empty:
        affordable = pool.nsmallest(20, "estimated_price_idr").copy()
    affordable["_score"]   = affordable.apply(lambda r: _score_food(r.to_dict(), goal, meal_type, budget_tier), axis=1)
    affordable["_cal_eff"] = affordable["calories_per_portion"] / affordable["estimated_price_idr"].clip(lower=500) * 1000
    if budget_tier == "LOW":
        affordable = affordable.sort_values("_cal_eff", ascending=False)
    else:
        affordable["_combined"] = affordable["_cal_eff"] * 0.4 + affordable["_score"] * 0.6
        affordable = affordable.sort_values("_combined", ascending=False)
    cal_min, cal_max = calorie_target * 0.85, calorie_target * 1.15
    selected, rem_budget, accum_cal, used = [], budget_hard, 0, set()
    # Pass 1: greedy
    for idx, food in affordable.iterrows():
        if len(selected) >= max_items or accum_cal >= cal_min: break
        if food["estimated_price_idr"] > rem_budget or idx in used: continue
        if accum_cal + food["calories_per_portion"] > cal_max * 1.15: continue
        item = {"food_id": str(food.get("id", idx)), "name": str(food.get("name", "Unknown")),
                "servings": 1.0, "calories": float(food["calories_per_portion"]),
                "protein_g": float(food.get("protein_g", 0)), "fat_g": float(food.get("fat_g", 0)),
                "carbs_g": float(food.get("carbs_g", 0)), "cost_idr": int(food["estimated_price_idr"]),
                "category": str(food.get("category", "STAPLE"))}
        selected.append(item); rem_budget -= item["cost_idr"]
        accum_cal += item["calories"]; used.add(idx)
    # Pass 2: fill gap
    if accum_cal < cal_min and len(selected) < max_items:
        needed = cal_min - accum_cal
        cands  = affordable[(~affordable.index.isin(used)) &
                             (affordable["estimated_price_idr"] <= rem_budget) &
                             (affordable["calories_per_portion"] >= needed * 0.2) &
                             (affordable["calories_per_portion"] <= needed * 2.0)].copy()
        if not cands.empty:
            best = cands.assign(_gap=lambda d: abs(d["calories_per_portion"] - needed)).nsmallest(1, "_gap").iloc[0]
            item = {"food_id": str(best.get("id", best.name)), "name": str(best.get("name", "Unknown")),
                    "servings": 1.0, "calories": float(best["calories_per_portion"]),
                    "protein_g": float(best.get("protein_g", 0)), "fat_g": float(best.get("fat_g", 0)),
                    "carbs_g": float(best.get("carbs_g", 0)), "cost_idr": int(best["estimated_price_idr"]),
                    "category": str(best.get("category", "STAPLE"))}
            selected.append(item); accum_cal += item["calories"]; rem_budget -= item["cost_idr"]
    # Pass 3: fractional
    if selected:
        tot = sum(s["calories"] for s in selected)
        dev = abs(tot - calorie_target) / max(calorie_target, 1)
        if dev > 0.15:
            last = selected[-1]
            if last["calories"] > 0:
                adj = max(0.5, min(1.5, 1 - (tot - calorie_target) / last["calories"]))
                adj = round(adj, 2)
                for k in ["calories", "protein_g", "fat_g", "carbs_g"]:
                    last[k] = round(last[k] * adj, 1)
                last["cost_idr"] = int(last["cost_idr"] * adj)
                last["servings"] = adj
    return selected

print("Knapsack v3 (multi-pass) ready")

Knapsack v3 (multi-pass) ready


In [26]:
# ── Meal Day Composer & Week Plan ────────────────────────────────
def compose_meal_day(profile, day_index, food_master, exclude_ids=None):
    exclude_ids = exclude_ids or set()
    goal        = profile.get("goal", "MAINTENANCE")
    daily_budget= profile.get("budget_per_day_idr", 35000)
    daily_cal   = profile.get("target_calories_per_day", 1900)
    meal_freq   = profile.get("meal_frequency", 3)
    restrictions= profile.get("diet_restrictions", [])
    tier        = _get_budget_tier(daily_budget)
    pool = food_master[food_master["is_active"] == True].copy()
    for r in restrictions:
        r = r.upper()
        if r == "HALAL" and "is_halal" in pool.columns:       pool = pool[pool["is_halal"] == True]
        elif r == "VEGETARIAN" and "is_vegetarian" in pool.columns: pool = pool[pool["is_vegetarian"] == True]
        elif r == "VEGAN" and "is_vegan" in pool.columns:     pool = pool[pool["is_vegan"] == True]
    if exclude_ids:
        pool = pool[~pool.index.isin(exclude_ids)]
    splits    = MEAL_SPLITS.get(meal_freq, MEAL_SPLITS[3])
    meals     = []
    day_used  = set()
    for meal_type, ratio in splits.items():
        meal_budget = int(daily_budget * ratio)
        meal_cal    = int(daily_cal * ratio)
        meal_pool   = pool[~pool.index.isin(day_used)]
        max_items   = 3 if tier == "LOW" else (4 if tier == "MID" else 5)
        foods = knapsack_v3(meal_pool, meal_budget, meal_cal,
                            goal=goal, meal_type=meal_type, budget_tier=tier, max_items=max_items)
        for f in foods: day_used.add(f["food_id"])
        meals.append({"meal_type": meal_type,
                      "calories": round(sum(f["calories"] for f in foods), 1),
                      "target_calories": meal_cal,
                      "cost_idr": sum(f["cost_idr"] for f in foods),
                      "budget_idr": meal_budget,
                      "protein_g": round(sum(f["protein_g"] for f in foods), 1),
                      "fat_g": round(sum(f["fat_g"] for f in foods), 1),
                      "carbs_g": round(sum(f["carbs_g"] for f in foods), 1),
                      "foods": foods})
    total_cal  = sum(m["calories"] for m in meals)
    total_cost = sum(m["cost_idr"] for m in meals)
    cal_dev    = abs(total_cal - daily_cal) / max(daily_cal, 1) * 100
    return {"day_index": day_index, "total_calories": round(total_cal, 1),
            "target_calories": daily_cal, "calorie_deviation_pct": round(cal_dev, 2),
            "total_cost_idr": total_cost, "budget_idr": daily_budget,
            "budget_ok": bool(total_cost <= daily_budget), "budget_tier": tier,
            "total_protein_g": round(sum(m["protein_g"] for m in meals), 1),
            "total_fat_g": round(sum(m["fat_g"] for m in meals), 1),
            "total_carbs_g": round(sum(m["carbs_g"] for m in meals), 1),
            "meals": meals}

def compose_week_plan(profile, food_master):
    days, weekly_used = [], set()
    for d in range(7):
        day = compose_meal_day(profile, d, food_master, exclude_ids=weekly_used)
        days.append(day)
        for m in day["meals"]:
            for f in m["foods"]:
                if f.get("category") not in ("SNACK", "BEVERAGE"):
                    weekly_used.add(f["food_id"])
        tier = _get_budget_tier(profile.get("budget_per_day_idr", 35000))
        if (d + 1) % (2 if tier == "LOW" else 3) == 0:
            weekly_used = set()
    return days

print("Composer ready")

Composer ready


In [27]:
# ── Validation: 5 Budget Levels x 7 Days ─────────────────────────
TEST_PROFILES = [
    {"name": "Ultra Low (10K) — Weight Loss",  "goal": "WEIGHT_LOSS",  "budget_per_day_idr": 10000, "target_calories_per_day": 1400, "meal_frequency": 2, "diet_restrictions": ["HALAL"]},
    {"name": "Low Budget (15K) — Maintenance", "goal": "MAINTENANCE",  "budget_per_day_idr": 15000, "target_calories_per_day": 1600, "meal_frequency": 3, "diet_restrictions": ["HALAL"]},
    {"name": "Budget (25K) — Weight Loss",     "goal": "WEIGHT_LOSS",  "budget_per_day_idr": 25000, "target_calories_per_day": 1700, "meal_frequency": 3, "diet_restrictions": ["HALAL"]},
    {"name": "Mid Budget (40K) — Maintenance", "goal": "MAINTENANCE",  "budget_per_day_idr": 40000, "target_calories_per_day": 1900, "meal_frequency": 3, "diet_restrictions": ["HALAL"]},
    {"name": "High Budget (75K) — Muscle Gain","goal": "MUSCLE_GAIN",  "budget_per_day_idr": 75000, "target_calories_per_day": 2500, "meal_frequency": 4, "diet_restrictions": ["HALAL"]},
]

all_devs, all_budget_ok, total_days, all_results = [], 0, 0, {}
print(f"{'═'*60}\nVALIDATION: 5 Budget Levels x 7 Days\n{'═'*60}")

for prof in TEST_PROFILES:
    budget = prof["budget_per_day_idr"]
    tier   = _get_budget_tier(budget)
    print(f"\nPROFIL: {prof['name']} [Tier: {tier}]")
    print(f"  Budget: Rp{budget:,}/hari | Target: {prof['target_calories_per_day']} kkal")
    days = compose_week_plan(prof, df_food)
    p_devs, p_ok = [], 0
    for day in days:
        dev  = day["calorie_deviation_pct"]
        p_devs.append(dev); total_days += 1
        if day["budget_ok"]: p_ok += 1
        st = "OK" if dev <= 15 else ("!!" if dev <= 25 else "XX")
        bs = "OK" if day["budget_ok"] else "OV"
        print(f"  Day {day['day_index']+1}: {day['total_calories']:>7.0f} kkal (dev {dev:>5.1f}% {st}) | Rp{day['total_cost_idr']:>6,} {bs} | P:{day['total_protein_g']:.0f}g F:{day['total_fat_g']:.0f}g C:{day['total_carbs_g']:.0f}g")
    avg_dev = float(np.mean(p_devs)); max_dev = float(np.max(p_devs))
    all_devs.extend(p_devs); all_budget_ok += p_ok
    print(f"  Avg dev: {avg_dev:.1f}% | Max dev: {max_dev:.1f}% | Budget OK: {p_ok}/7")
    all_results[prof["name"]] = {"budget_per_day": budget, "budget_tier": tier,
                                  "avg_cal_deviation": round(avg_dev, 2), "max_cal_deviation": round(max_dev, 2),
                                  "budget_compliance": f"{p_ok}/7"}

g_avg = float(np.mean(all_devs)); g_max = float(np.max(all_devs))
print(f"\n{'═'*60}\nOVERALL ({total_days} days)\n{'═'*60}")
print(f"  Avg deviation  : {g_avg:.1f}%")
print(f"  Max deviation  : {g_max:.1f}%")
print(f"  Budget OK      : {all_budget_ok}/{total_days}")
print(f"  Target ≤15%    : {'PASS' if g_avg <= 15 else 'FAIL'}")

════════════════════════════════════════════════════════════
VALIDATION: 5 Budget Levels x 7 Days
════════════════════════════════════════════════════════════

PROFIL: Ultra Low (10K) — Weight Loss [Tier: LOW]
  Budget: Rp10,000/hari | Target: 1400 kkal
  Day 1:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Day 2:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Day 3:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Day 4:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Day 5:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Day 6:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Day 7:    1050 kkal (dev  25.0% !!) | Rp15,979 OV | P:24g F:54g C:118g
  Avg dev: 25.0% | Max dev: 25.0% | Budget OK: 0/7

PROFIL: Low Budget (15K) — Maintenance [Tier: LOW]
  Budget: Rp15,000/hari | Target: 1600 kkal
  Day 1:    1497 kkal (dev   6.4% OK) | Rp23,004 OV | P:46g F:80g C:150g
  Day 2:    149

In [29]:
# ── Save outputs ──────────────────────────────────────────────────
report = {
    "version": "v3-dynamic-budget",
    "min_budget_idr": 10000,
    "calorie_deviation_avg_pct": round(g_avg, 2),
    "calorie_deviation_max_pct": round(g_max, 2),
    "budget_compliance": f"{all_budget_ok}/{total_days}",
    "target_met": bool(g_avg <= 15),
    "improvement_vs_v1": {"v1_avg_deviation": 48.74, "v3_avg_deviation": round(g_avg, 2)},
    "per_profile": all_results,
}
with open("output/validation/validation_report_v3.json", "w") as f:
    json.dump(report, f, indent=2)
print("Saved: output/validation/validation_report_v3.json")

config = {
    "version": "3.0-dynamic-budget",
    "min_budget_idr": 10000,
    "goal_weights": GOAL_WEIGHTS,
    "meal_splits": MEAL_SPLITS,
    "budget_tiers": {"LOW": "<20K", "MID": "20-50K", "HIGH": ">50K"},
    "low_budget_relaxation": "20% budget overshoot allowed",
    "calorie_tolerance": 0.15,
}
with open("output/models/knapsack_config_v3.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved: output/models/knapsack_config_v3.json")

Saved: output/validation/validation_report_v3.json
Saved: output/models/knapsack_config_v3.json
